# 03. 과적합과 일반화 — 함정을 직접 재현하기

**대응 강의:** [4강 모델의 일반화](../../Lecture/03-머신러닝-기본개념/04-모델의-일반화.md)

머신러닝에서 제일 흔하게 빠지는 함정이 '과적합'이에요. 이 노트북에서는 그 함정을 일부러 만들어 보면서 감을 잡아볼게요. 같이 해볼 건:
- 다항식 차수를 점점 올려가며 **과소적합 → 적절 → 과적합**으로 변하는 모습 눈으로 보기
- 훈련 점수와 테스트 점수가 **벌어지는 차이**로 과적합을 진단하기
- **데이터 누수(data leakage)** 가 점수를 어떻게 부풀리는지 직접 재현하기

> '훈련을 잘 맞히는 것'과 '진짜 실력'은 다르다는 걸 몸으로 느껴 보는 노트북이에요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

# 숨어 있는 진짜 관계는 부드러운 곡선(cos)이에요. 거기에 노이즈를 섞어서 진짜 데이터처럼 만들어요
rng = np.random.default_rng(0)
def true_fn(x):
    return np.cos(1.5 * np.pi * x)

n = 30
X = np.sort(rng.uniform(0, 1, n))
y = true_fn(X) + rng.normal(0, 0.15, n)
X = X.reshape(-1, 1)
print('샘플 수:', n)

## 1. 차수를 올리면: 과소적합 → 적절 → 과적합

차수 1(직선)은 너무 단순해서 데이터를 못 따라가요(과소적합). 반대로 차수 15는 노이즈까지 통째로 외워 버려요(과적합). 그 사이 어딘가에 딱 좋은 지점이 있어요.

In [ ]:
degrees = [1, 4, 15]
titles = ['Underfit (too simple)', 'Good fit', 'Overfit (memorizing noise)']
x_plot = np.linspace(0, 1, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d, t in zip(axes, degrees, titles):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X, y)
    ax.scatter(X, y, color='k', s=20, label='data')
    ax.plot(x_plot, true_fn(x_plot), 'g--', alpha=0.6, label='true function')
    ax.plot(x_plot, model.predict(x_plot), 'r-', label=f'degree {d} model')
    ax.set_ylim(-2, 2); ax.set_title(t); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. 훈련 오차 vs 테스트 오차로 과적합 잡아내기

차수를 1부터 15까지 올려가면서, **훈련 오차**와 **테스트 오차**를 따로 그려 볼게요.

- 훈련 오차는 차수가 올라갈수록 계속 줄어들어요 (데이터를 외워 버리니까요).
- 그런데 테스트 오차는 어느 순간부터 **다시 올라가요** → 바로 그 지점이 과적합이 시작되는 곳이에요!

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=1)

max_deg = 15
train_err, test_err = [], []
for d in range(1, max_deg + 1):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression())
    m.fit(X_tr, y_tr)
    train_err.append(mean_squared_error(y_tr, m.predict(X_tr)))
    test_err.append(mean_squared_error(y_te, m.predict(X_te)))

plt.figure(figsize=(8, 5))
plt.plot(range(1, max_deg + 1), train_err, 'o-', label='train error')
plt.plot(range(1, max_deg + 1), test_err, 's-', label='test error')
plt.yscale('log')
plt.axvspan(8, max_deg, alpha=0.1, color='red')
plt.text(11, max(test_err), 'overfit zone', color='red')
plt.xlabel('polynomial degree (model complexity)'); plt.ylabel('error (MSE, log)')
plt.title('More complexity: train improves but test breaks down')
plt.legend(); plt.show()

best = int(np.argmin(test_err)) + 1
print('테스트 오차가 가장 낮은 차수(적절한 복잡도):', best)

## 3. 데이터 누수(Data Leakage) 재현 — 점수를 부풀리는 함정

스케일링을 **데이터를 나누기 전에** 전체로 해 버리면, 훈련 과정이 테스트 데이터의 정보를 미리 훔쳐보게 돼요.
올바른 방법(먼저 나누고, 훈련 데이터로만 fit)과 점수를 나란히 비교해 볼게요.

> 차이를 확실하게 보여 주려고, 일부러 샘플은 적고 특성은 많은 극단적인 상황을 만들었어요.

In [ ]:
from sklearn.linear_model import Ridge

# 타깃과 아무 관계 없는 순수 노이즈 특성들 (정상이라면 성능이 형편없이 나와야 맞아요)
rng2 = np.random.default_rng(42)
n_samples, n_features = 60, 200
Xn = rng2.normal(size=(n_samples, n_features))
yn = rng2.normal(size=n_samples)   # 특성이랑 전혀 상관없는 타깃이에요!

# ❌ 잘못된 방법: 전체 데이터로 먼저 스케일링하고 나서 나누기 (테스트 정보가 새어 들어가요)
Xn_leak = StandardScaler().fit_transform(Xn)
Xtr, Xte, ytr, yte = train_test_split(Xn_leak, yn, test_size=0.3, random_state=0)
leak_model = Ridge().fit(Xtr, ytr)
leak_score = leak_model.score(Xte, yte)

# ✅ 올바른 방법: 먼저 나누고 → 훈련 데이터로만 스케일러를 fit
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xn, yn, test_size=0.3, random_state=0)
sc = StandardScaler().fit(Xtr2)          # 훈련 데이터로만 학습
clean_model = Ridge().fit(sc.transform(Xtr2), ytr2)
clean_score = clean_model.score(sc.transform(Xte2), yte2)

print('타깃이 특성과 무관하니까, 제대로 했다면 R^2는 0 근처(혹은 음수)가 정상이에요\n')
print(f'❌ 누수 버전 테스트 R^2 : {leak_score:.3f}  <- 비현실적으로 높으면 누수를 의심하세요!')
print(f'✅ 정상 버전 테스트 R^2 : {clean_score:.3f}  <- 이게 현실이에요 (관계가 없으니 낮게 나와요)')

> 💡 위에서 누수 버전 점수가 정상 버전보다 부자연스럽게 높게 나왔죠?
> 실전에서도 **"점수가 너무 잘 나오면"** 좋아하기 전에 누수부터 의심하는 게 좋아요.
>
> **예방책:** scikit-learn `Pipeline` 안에 스케일러를 넣어 두면, 교차검증을 할 때 각 분할의
> 훈련 데이터로만 알아서 fit 돼서 누수가 구조적으로 막혀요.

## 정리

- 모델이 복잡해질수록 훈련 오차는 줄지만, 테스트 오차는 어느 순간부터 다시 늘어나요(과적합).
- 과적합 진단법 = **훈련 점수와 테스트 점수가 얼마나 벌어졌는지** 보기.
- 데이터 누수는 점수를 비현실적으로 부풀려요. 그러니 **나누는 걸 항상 제일 먼저** 하세요.

## 🎯 직접 해보기

1. Part 1에서 차수 리스트를 `[2, 6, 25]`로 바꾸면 그림이 어떻게 달라지는지 보세요.
2. Part 2에서 노이즈 크기(`0.15`)를 `0.4`로 키우면 '적절한 차수'가 어디로 옮겨가나요?
3. Part 3을 `make_pipeline(StandardScaler(), Ridge())` + `cross_val_score`로 다시 짜서, 파이프라인이 정말 누수를 막아 주는지 확인해 보세요.